# Sentiment Scoring 

## Imports

Load numeric and dataframe utilities.

In [1]:
import numpy as np
import pandas as pd

## Load Transformer Components

Bring in tokenizer and model classes from Hugging Face Transformers.

In [2]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

/Users/dylanhuang/micromamba/envs/df_ae2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Torch Setup

Import PyTorch for model inference.

In [3]:
import torch

## Load Tweets

Read the cleaned, non-merged tweet dataset.

In [4]:
tweets = pd.read_parquet("../data/dataset/stock_tweet_nomerge.parquet")

## Preview Tweets

Inspect a sample of the tweets before scoring.

In [5]:
tweets

,ticker,text,created_at,user_id,date
0,VZ,"gdp news "" actives on open aapl tsla twtr c gd...",2014-10-14 13:39:49+00:00,1923902360,2014-10-14
1,VZ,psw seeking alpha june trade review fxi ewj ir...,2015-06-20 14:18:10+00:00,3232889321,2015-06-20
2,VZ,psw seeking alpha june trade review fxi ewj ir...,2015-06-20 16:16:05+00:00,3238685909,2015-06-20
3,VZ,psw seeking alpha june trade review fxi ewj ir...,2015-06-20 16:25:57+00:00,2740607613,2015-06-20
4,VZ,psw seeking alpha june trade review fxi ewj ir...,2015-06-20 16:01:01+00:00,3234872187,2015-06-20
...,...,...,...,...,...
106333,CODI,stock contest pick googl and win a free tablet...,2014-09-07 00:20:34+00:00,386787305,2014-09-07
106334,CODI,compass diversified holdings wk low click here...,2015-05-18 13:34:15+00:00,2579477766,2015-05-18
106335,CODI,compass diversified holdings files sec form k ...,2014-08-09 10:38:34+00:00,2717670854,2014-08-09
106336,CODI,compass diversified holdings financials aapl i...,2015-06-01 19:34:12+00:00,2181403417,2015-06-01


## Dataset Size

Check the number of tweet rows to be scored.

In [6]:
tweets.shape

(106338, 5)

## Load Sentiment Model

Initialize the pretrained sentiment model and tokenizer.

In [7]:
tokenizer = AutoTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')
bert_model = AutoModelForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1393.29it/s, Materializing param=classifier.weight]                                     


## Select Device

Move the model to GPU if available, otherwise use CPU.

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model = bert_model.to(device)

## Confirm Device

Verify which device the model is running on.

In [9]:
torch.cuda.is_available()
print(next(bert_model.parameters()).device)

cpu


## Sentiment Scoring Function

Define a helper that returns a 1–5 sentiment score from model logits.

In [10]:
def cal_sentiment_score(text):
    inputs = tokenizer.encode(text, return_tensors='pt', truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        outputs = bert_model(inputs)
    
    sentiment = int(torch.argmax(outputs.logits)) + 1
    return sentiment

## Score All Tweets

Apply the sentiment model to each tweet.

In [11]:
tweets['sentiment'] = tweets['text'].apply(cal_sentiment_score)

## Preview Scored Tweets

Inspect the sentiment column after scoring.

In [12]:
tweets

,ticker,text,created_at,user_id,date,sentiment
0,VZ,"gdp news "" actives on open aapl tsla twtr c gd...",2014-10-14 13:39:49+00:00,1923902360,2014-10-14,1
1,VZ,psw seeking alpha june trade review fxi ewj ir...,2015-06-20 14:18:10+00:00,3232889321,2015-06-20,5
2,VZ,psw seeking alpha june trade review fxi ewj ir...,2015-06-20 16:16:05+00:00,3238685909,2015-06-20,5
3,VZ,psw seeking alpha june trade review fxi ewj ir...,2015-06-20 16:25:57+00:00,2740607613,2015-06-20,5
4,VZ,psw seeking alpha june trade review fxi ewj ir...,2015-06-20 16:01:01+00:00,3234872187,2015-06-20,5
...,...,...,...,...,...,...
106333,CODI,stock contest pick googl and win a free tablet...,2014-09-07 00:20:34+00:00,386787305,2014-09-07,5
106334,CODI,compass diversified holdings wk low click here...,2015-05-18 13:34:15+00:00,2579477766,2015-05-18,5
106335,CODI,compass diversified holdings files sec form k ...,2014-08-09 10:38:34+00:00,2717670854,2014-08-09,5
106336,CODI,compass diversified holdings financials aapl i...,2015-06-01 19:34:12+00:00,2181403417,2015-06-01,5


## Save Output

Write the scored dataset to parquet for downstream feature engineering.

In [13]:
tweets.to_parquet('../data/dataset/stock_tweets_sentiment_nomerge.parquet',index=False)